# Agentomics - a PydanticAI-powered autonomous ML Agent

In [ ]:
%%capture
!pip install -q condacolab pydantic-ai
import condacolab
condacolab.install()

In [3]:
!git clone https://github.com/BioGeMT/agentomics-ml.git
!./agentomics-ml/scripts/download_example_dataset.sh --dataset "AGO2_CLASH_Hejret2023"

Cloning into 'agentomics-ml'...
remote: Enumerating objects: 9687, done.
remote: Counting objects: 100% (3624/3624), done.
remote: Compressing objects: 100% (1050/1050), done.
remote: Total 9687 (delta 2855), reused 2623 (delta 2574), pack-reused 6063 (from 2)
Receiving objects: 100% (9687/9687), 70.15 MiB | 4.05 MiB/s, done.
Resolving deltas: 100% (6612/6612), done.
2 channel Terms of Service accepted

Remove all packages in environment /opt/homebrew/Caskroom/miniconda/base/envs/agentomics-datasets:


## Package Plan ##

  environment location: /opt/homebrew/Caskroom/miniconda/base/envs/agentomics-datasets


The following packages will be REMOVED:

  _openmp_mutex-4.5-7_kmp_llvm
  bzip2-1.0.8-hd037594_9
  ca-certificates-2026.5.20-hbd8a1cb_0
  icu-78.3-hef89b57_0
  joblib-1.5.3-pyhd8ed1ab_0
  lcms2-2.19.1-hdfa7624_1
  lerc-4.1.0-h1eee2c3_0
  libblas-3.11.0-8_h51639a9_openblas
  libcblas-3.11.0-8_hb0561ab_openblas
  libcxx-22.1.7-h55c6f16_0
  libdeflate-1.25-hc11a715_0
  libexpat-2.8.1

In [ ]:
from pathlib import Path
openrouter_api_key = ""  # Paste the workshop key here, for example: sk-or-v1-...

env_path = Path("agentomics-ml") / ".env"
env_path.write_text(f"OPENROUTER_API_KEY={openrouter_api_key}\n")

## Run Agentomics

In [ ]:
!./agentomics-ml/run.sh --local --dataset AGO2_CLASH_Hejret2023 --model openai/gpt-5.4-mini --iterations 1 --val-metric AUROC --cpu-only

# Pydantic-AI connection

In [ ]:
# Example of Structured output used in Agentomics
from pydantic import BaseModel, Field
from pydantic_ai import ModelRetry
import os

class ModelTrainingOutput(BaseModel):
    path_to_train_file: str = Field(description="Absolute path to the generated 'train.py'")
    path_to_model_file: str = Field(description="Absolute path to the trained model file")
    path_to_artifacts_dir: str = Field(
        description=(
            "Absolute path to the folder with artifacts produced by training. "
            "Must be called 'training_artifacts'. "
            "(This folder should be the parent of path_to_model_file and a sibling to train.py)"
        )
    )
    training_summary: str = Field(
        description="Short summary of the training implementation. Don't include any metrics in this summary."
    )
    unresolved_issues: str | None = Field(
        description=(
            "Issues that remain unresolved and could impact performance and/or metrics. "
            "(e.g. expected GPU to be available but is inaccessible during training, "
            "foundation model could not be loaded, etc...). Can be empty."
        )
    )
    
    

In [ ]:
# How we validate the structured output for model training?

@model_training_agent.output_validator
def validate_model_training_output(result: dict) -> ModelTrainingOutput:
    if not os.path.exists(result.path_to_train_file):
        raise ModelRetry(f"Train file does not exist. {result.path_to_train_file}")
    if Path(result.path_to_train_file).name.strip() != "train.py":
        raise ModelRetry(f"Train file must be called 'train.py' , currently is named {Path(result.path_to_train_file).name.strip()}")
    if not os.path.exists(result.path_to_model_file):
        raise ModelRetry(f"Model file does not exist at {result.path_to_model_file}")